In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_io as tfio
from tensorflow import keras
from tensorflow.keras import layers

# =============================
# CONFIGURATION
# =============================
RUN_NAME = "inceptionv3_299_fulloutput_csvsplit"

IMAGE_DIR = "/home/spaulgupta2000/data/pcam_raw/train_full_output" #trainning images with 4xultrasharp
TEST_IMAGE_DIR = "/home/spaulgupta2000/data/pcam_raw/test_full_output"# testing images with 4x Ultrasharp

TRAIN_CSV = "/home/spaulgupta2000/data/pcam_raw/split_train_80.csv" # true label  for training image
TEST_CSV = "/home/spaulgupta2000/data/pcam_raw/split_test_80.csv" #true labeled test images for statistical analysis

IMG_SIZE = (299, 299)
BATCH_SIZE = 32
SEED = 42

# Stage 1: frozen backbone
EPOCHS_STAGE1 = 8
LR_STAGE1 = 1e-3

# Stage 2: fine-tuning
EPOCHS_STAGE2 = 12
LR_STAGE2 = 1e-5
UNFREEZE_LAST_LAYERS = 80

VAL_SPLIT_FROM_TRAIN = 0.2

# You can keep "val_loss" or change to "val_auc"
MONITOR = "val_loss"
MODE = "min" if MONITOR == "val_loss" else "max"

# =============================
# SAVE PATHS
# =============================
SAVE_DIR = f"/home/spaulgupta2000/results_{RUN_NAME}"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_H5 = os.path.join(SAVE_DIR, f"{RUN_NAME}_best.h5")
FINAL_H5 = os.path.join(SAVE_DIR, f"{RUN_NAME}_final.h5")

NPY_TRUE = os.path.join(SAVE_DIR, f"y_true_{RUN_NAME}.npy")
NPY_PRED = os.path.join(SAVE_DIR, f"y_pred_{RUN_NAME}.npy")
NPY_PROB = os.path.join(SAVE_DIR, f"y_prob_{RUN_NAME}.npy")

METRICS_JSON = os.path.join(SAVE_DIR, f"metrics_{RUN_NAME}.json")
HISTORY_JSON = os.path.join(SAVE_DIR, f"history_{RUN_NAME}.json")

HISTORY_STAGE1_NPY = os.path.join(SAVE_DIR, f"history_stage1_{RUN_NAME}.npy")
HISTORY_STAGE2_NPY = os.path.join(SAVE_DIR, f"history_stage2_{RUN_NAME}.npy")
CSV_LOG = os.path.join(SAVE_DIR, f"training_log_{RUN_NAME}.csv")

# =============================
# REPRODUCIBILITY
# =============================
tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

# =============================
# HELPERS
# =============================
def safe_print(*args, **kwargs):
    print(*args, **kwargs, flush=True)

def save_json(data, path):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(data, f, indent=4)
    os.replace(tmp_path, path)

def add_extension_if_missing(name):
    name = str(name)
    if name.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
        return name
    return name + ".tif"

def detect_columns(df):
    filename_col = None
    label_col = None

    for c in df.columns:
        cl = c.lower()
        if cl in ["filename", "file_name", "image", "image_id", "id", "path"]:
            filename_col = c
            break

    for c in df.columns:
        cl = c.lower()
        if cl in ["label", "target", "class", "y"]:
            label_col = c
            break

    if filename_col is None or label_col is None:
        raise ValueError(
            f"Could not detect filename/label columns from columns: {df.columns.tolist()}"
        )

    return filename_col, label_col

def make_full_paths(df, file_col, base_dir):
    filenames = df[file_col].astype(str).apply(add_extension_if_missing).tolist()
    return [os.path.join(base_dir, f) for f in filenames]

def merge_histories(hist1, hist2):
    merged = {}
    all_keys = set(hist1.keys()) | set(hist2.keys())
    for k in sorted(all_keys):
        v1 = hist1.get(k, [])
        v2 = hist2.get(k, [])
        merged[k] = [float(x) for x in v1] + [float(x) for x in v2]
    return merged

# =============================
# STARTUP INFO
# =============================
safe_print("TensorFlow version:", tf.__version__)
safe_print("Using GPU:", tf.config.list_physical_devices("GPU"))
safe_print("IMAGE_DIR     :", IMAGE_DIR)
safe_print("TEST_IMAGE_DIR:", TEST_IMAGE_DIR)
safe_print("TRAIN_CSV     :", TRAIN_CSV)
safe_print("TEST_CSV      :", TEST_CSV)
safe_print("SAVE_DIR      :", SAVE_DIR)

for p in [IMAGE_DIR, TEST_IMAGE_DIR, TRAIN_CSV, TEST_CSV]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Not found: {p}")

# =============================
# LOAD CSV SPLITS
# =============================
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

safe_print("\nTrain CSV columns:", train_df.columns.tolist())
safe_print("Test CSV columns :", test_df.columns.tolist())

train_file_col, train_label_col = detect_columns(train_df)
test_file_col, test_label_col = detect_columns(test_df)

safe_print(f"Detected train columns -> file: {train_file_col}, label: {train_label_col}")
safe_print(f"Detected test columns  -> file: {test_file_col}, label: {test_label_col}")

# =============================
# TRAIN / VAL SPLIT
# =============================
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

val_size = int(len(train_df) * VAL_SPLIT_FROM_TRAIN)
val_df = train_df.iloc[:val_size].reset_index(drop=True)
train_df = train_df.iloc[val_size:].reset_index(drop=True)

safe_print(f"\nTrain samples: {len(train_df)}")
safe_print(f"Val samples  : {len(val_df)}")
safe_print(f"Test samples : {len(test_df)}")

# =============================
# FILEPATHS + LABELS
# =============================
train_paths = make_full_paths(train_df, train_file_col, IMAGE_DIR)
val_paths = make_full_paths(val_df, train_file_col, IMAGE_DIR)
test_paths = make_full_paths(test_df, test_file_col, TEST_IMAGE_DIR)

train_labels = train_df[train_label_col].astype("float32").values
val_labels = val_df[train_label_col].astype("float32").values
test_labels = test_df[test_label_col].astype("float32").values

missing_train = sum(not os.path.exists(p) for p in train_paths)
missing_val = sum(not os.path.exists(p) for p in val_paths)
missing_test = sum(not os.path.exists(p) for p in test_paths)

safe_print(f"Missing train files: {missing_train}")
safe_print(f"Missing val files  : {missing_val}")
safe_print(f"Missing test files : {missing_test}")

if missing_train > 0 or missing_val > 0 or missing_test > 0:
    safe_print("Examples of missing files:")
    for p in train_paths[:5] + val_paths[:5] + test_paths[:5]:
        if not os.path.exists(p):
            safe_print("  Missing ->", p)

    raise FileNotFoundError("Some image files listed in CSV do not exist. Fix the paths before training.")

# =============================
# DATASET BUILDER
# =============================
def decode_tiff_to_rgb(image_bytes):
    image = tfio.experimental.image.decode_tiff(image_bytes)
    image = tf.cast(image, tf.float32)

    # Ensure rank-3
    image = tf.ensure_shape(image, [None, None, None])

    channels = tf.shape(image)[-1]

    def gray_to_rgb():
        return tf.image.grayscale_to_rgb(image)

    def keep_first_three():
        return image[..., :3]

    def keep_as_is():
        return image

    image = tf.cond(
        tf.equal(channels, 1),
        gray_to_rgb,
        lambda: tf.cond(tf.greater(channels, 3), keep_first_three, keep_as_is)
    )
    return image

def load_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = decode_tiff_to_rgb(image_bytes)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    label = tf.cast(label, tf.float32)
    label = tf.reshape(label, (1,))
    return image, label

def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels, training=False)
test_ds = make_dataset(test_paths, test_labels, training=False)

class_names = ["0", "1"]
safe_print("Class order:", class_names)

# =============================
# DATA AUGMENTATION
# =============================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

# =============================
# MODEL
# =============================
base_model = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,), name="input_image")
x = data_augmentation(inputs)
x = tf.keras.applications.inception_v3.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid", name="prediction")(x)

model = keras.Model(inputs, outputs, name=RUN_NAME)

# =============================
# COMPILE FUNCTION
# =============================
def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ]
    )

compile_model(model, LR_STAGE1)
model.summary(print_fn=lambda x: safe_print(x))

# =============================
# CUSTOM CALLBACKS
# =============================
class EpochSummaryPrinter(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        safe_print(
            f"Epoch {epoch + 1:03d} | "
            f"loss={logs.get('loss', float('nan')):.6f} | "
            f"accuracy={logs.get('accuracy', float('nan')):.6f} | "
            f"auc={logs.get('auc', float('nan')):.6f} | "
            f"val_loss={logs.get('val_loss', float('nan')):.6f} | "
            f"val_accuracy={logs.get('val_accuracy', float('nan')):.6f} | "
            f"val_auc={logs.get('val_auc', float('nan')):.6f}"
        )

class HistoryJSONLogger(keras.callbacks.Callback):
    def __init__(self, path):
        super().__init__()
        self.path = path
        self.history_data = {"epochs": []}

    def on_train_begin(self, logs=None):
        if os.path.exists(self.path):
            try:
                with open(self.path, "r") as f:
                    self.history_data = json.load(f)
            except Exception:
                self.history_data = {"epochs": []}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        epoch_record = {"epoch": int(epoch + 1)}
        for k, v in logs.items():
            try:
                epoch_record[k] = float(v)
            except Exception:
                pass

        self.history_data["epochs"].append(epoch_record)
        save_json(self.history_data, self.path)

history_json_logger = HistoryJSONLogger(HISTORY_JSON)
epoch_printer = EpochSummaryPrinter()
csv_logger = keras.callbacks.CSVLogger(CSV_LOG, append=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=BEST_H5,
        monitor=MONITOR,
        mode=MODE,
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor=MONITOR,
        mode=MODE,
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.TerminateOnNaN(),
    csv_logger,
    history_json_logger,
    epoch_printer,
]

# =============================
# STAGE 1
# =============================
safe_print("\n==================== STAGE 1: TRAIN HEAD ====================")
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks,
    verbose=0
)

np.save(HISTORY_STAGE1_NPY, history_stage1.history, allow_pickle=True)
safe_print("Saved stage1 history backup:", HISTORY_STAGE1_NPY)

# =============================
# STAGE 2
# =============================
safe_print("\n==================== STAGE 2: FINE-TUNING ====================")
base_model.trainable = True

for layer in base_model.layers[:-UNFREEZE_LAST_LAYERS]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

compile_model(model, LR_STAGE2)

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1 + EPOCHS_STAGE2,
    initial_epoch=EPOCHS_STAGE1,
    callbacks=callbacks,
    verbose=0
)

np.save(HISTORY_STAGE2_NPY, history_stage2.history, allow_pickle=True)
safe_print("Saved stage2 history backup:", HISTORY_STAGE2_NPY)

# =============================
# SAVE FINAL MODEL
# =============================
model.save(FINAL_H5)
safe_print("\nSaved final model:", FINAL_H5)
safe_print("Saved best model :", BEST_H5)

# =============================
# COMBINED HISTORY SAVE
# =============================
combined_history = merge_histories(history_stage1.history, history_stage2.history)
combined_history["run_name"] = RUN_NAME
combined_history["csv_log"] = CSV_LOG
combined_history["best_model_path"] = BEST_H5
combined_history["final_model_path"] = FINAL_H5
combined_history["history_stage1_npy"] = HISTORY_STAGE1_NPY
combined_history["history_stage2_npy"] = HISTORY_STAGE2_NPY

save_json(combined_history, HISTORY_JSON)
safe_print("Saved combined history JSON:", HISTORY_JSON)

# =============================
# LOAD BEST MODEL FOR TESTING
# =============================
best_model = tf.keras.models.load_model(BEST_H5, compile=False)

# =============================
# TEST SET PREDICTION
# =============================
y_true_list = []
y_prob_list = []

for xb, yb in test_ds:
    probs = best_model.predict(xb, verbose=0).ravel()
    y_prob_list.append(probs)
    y_true_list.append(yb.numpy().ravel())

y_prob = np.concatenate(y_prob_list, axis=0)
y_true = np.concatenate(y_true_list, axis=0).astype(int)
y_pred = (y_prob >= 0.5).astype(int)

np.save(NPY_TRUE, y_true)
np.save(NPY_PRED, y_pred)
np.save(NPY_PROB, y_prob)

safe_print("Saved:", NPY_TRUE)
safe_print("Saved:", NPY_PRED)
safe_print("Saved:", NPY_PROB)

# =============================
# METRICS
# =============================
tn = int(np.sum((y_true == 0) & (y_pred == 0)))
fp = int(np.sum((y_true == 0) & (y_pred == 1)))
fn = int(np.sum((y_true == 1) & (y_pred == 0)))
tp = int(np.sum((y_true == 1) & (y_pred == 1)))

acc = (tp + tn) / max(tp + tn + fp + fn, 1)
precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-8)
specificity = tn / max(tn + fp, 1)

auc_metric = tf.keras.metrics.AUC()
auc_metric.update_state(y_true, y_prob)
auc = float(auc_metric.result().numpy())

safe_print("\n==================== TEST RESULTS ====================")
safe_print("Confusion Matrix [[TN, FP],[FN, TP]]:")
safe_print([[tn, fp], [fn, tp]])
safe_print(f"Accuracy    : {acc:.6f}")
safe_print(f"Precision   : {precision:.6f}")
safe_print(f"Recall/Sens.: {recall:.6f}")
safe_print(f"F1-score    : {f1:.6f}")
safe_print(f"Specificity : {specificity:.6f}")
safe_print(f"AUC         : {auc:.6f}")
safe_print("=====================================================\n")

results = {
    "run_name": RUN_NAME,
    "class_names": class_names,
    "img_size": list(IMG_SIZE),
    "batch_size": BATCH_SIZE,
    "epochs_stage1": EPOCHS_STAGE1,
    "epochs_stage2": EPOCHS_STAGE2,
    "learning_rate_stage1": LR_STAGE1,
    "learning_rate_stage2": LR_STAGE2,
    "unfreeze_last_layers": UNFREEZE_LAST_LAYERS,
    "train_csv": TRAIN_CSV,
    "test_csv": TEST_CSV,
    "save_dir": SAVE_DIR,
    "best_model_path": BEST_H5,
    "final_model_path": FINAL_H5,
    "history_json_path": HISTORY_JSON,
    "csv_log_path": CSV_LOG,
    "y_true_path": NPY_TRUE,
    "y_pred_path": NPY_PRED,
    "y_prob_path": NPY_PROB,
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp,
    "accuracy": acc,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "specificity": specificity,
    "auc": auc
}

save_json(results, METRICS_JSON)

safe_print("Saved metrics :", METRICS_JSON)
safe_print("Saved history :", HISTORY_JSON)
safe_print("Saved CSV log :", CSV_LOG)
safe_print("Done.")